# 02 - Prepare and export the data

Remove duplicate rows, create an 80/20 training/test split and export it to `data/processed/`. Notebooks 03 and 04 load these files. Category encoding and scaling are fitted later inside training folds.

## 2.1 Load the data

Read the original CSV and show its size and first few rows.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_folder = Path.cwd()
pd.set_option('display.max_columns', 30)

from sklearn.model_selection import train_test_split, StratifiedKFold
raw_data = pd.read_csv(project_folder / 'online_shoppers_intention.csv')
print('Rows and columns:', raw_data.shape)
display(raw_data.head())

Rows and columns: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


## 2.2 Data preparation decisions

Notebook 01 found no missing values or placeholder strings, so no imputation is needed. Keep large valid values. Remove exact duplicate rows before splitting.

## 2.3 Remove exact duplicates
Keep the first copy of each full row. The original zero-based row numbers stay in the index. Revenue=1 means purchase; Weekend stays 0/1.

In [2]:
clean_data = raw_data.drop_duplicates().copy()
clean_data['Revenue'] = clean_data['Revenue'].astype(int)
clean_data['Weekend'] = clean_data['Weekend'].astype(int)
print('Before:', len(raw_data), 'rows; purchase rate:', raw_data['Revenue'].mean())
print('After:', len(clean_data), 'rows; purchase rate:', clean_data['Revenue'].mean())

Before: 12330 rows; purchase rate: 0.15474452554744525
After: 12205 rows; purchase rate: 0.15632937320770177


## 2.4 Split into training and test data

Use 80% of the cleaned sessions for training and keep 20% for the final test. Stratification keeps a similar purchase proportion in both groups. Seed 42 makes the split repeatable.

Retain PageValues in the exports for the training sensitivity experiment. Main models exclude it because its availability before purchase is uncertain.

In [3]:
training_data, test_data = train_test_split(
    clean_data, test_size=0.20, stratify=clean_data['Revenue'], random_state=42)
display(pd.DataFrame({
    'sessions': [len(training_data), len(test_data)],
    'purchase_rate': [training_data['Revenue'].mean(), test_data['Revenue'].mean()],
}, index=['train', 'test']))

predictor_columns = training_data.columns.drop(['Revenue', 'PageValues']).tolist()
matching_pairs = training_data[predictor_columns].merge(test_data[predictor_columns])
print('Identical predictor pairs across train/test (excluding PageValues):', len(matching_pairs))

,sessions,purchase_rate
train,9764,0.156288
test,2441,0.156493


Identical predictor pairs across train/test (excluding PageValues):

 0


## 2.5 Preview five validation folds

Each training row is used for validation once. Notebooks 03 and 04 recreate these folds from the saved training-row order using the same seed.

In [4]:
validation_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_summary = []
for fold_number, (fit_rows, validation_rows) in enumerate(
        validation_folds.split(training_data, training_data['Revenue'])):
    fold_summary.append({
        'fold': fold_number, 'fit_sessions': len(fit_rows),
        'validation_sessions': len(validation_rows),
        'validation_purchase_rate': training_data.iloc[validation_rows]['Revenue'].mean()})
display(pd.DataFrame(fold_summary))

,fold,fit_sessions,validation_sessions,validation_purchase_rate
0,0,7811,1953,0.156682
1,1,7811,1953,0.156170
2,2,7811,1953,0.156170
3,3,7811,1953,0.156170
4,4,7812,1952,0.156250


## 2.6 Export the prepared data

Save original predictor values and the integer Revenue label without a row-index column. Rerunning this notebook replaces both files. Run it before notebooks 03 and 04.

Each model pipeline learns category encoding and any scaling from its training fold; the CSVs contain no fitted transformations.

In [5]:
output_folder = project_folder / 'data' / 'processed'
output_folder.mkdir(parents=True, exist_ok=True)
training_data.to_csv(output_folder / 'train.csv', index=False)
test_data.to_csv(output_folder / 'test.csv', index=False)
print('Saved:', output_folder / 'train.csv')
print('Saved:', output_folder / 'test.csv')

Saved: C:\code projects\ml-cw2\data\processed\train.csv
Saved: C:\code projects\ml-cw2\data\processed\test.csv
